In [1]:
from collections import deque
import heapq

# Graph Representation
graph = {
    'S': [('A', 2), ('B', 5), ('C', 1)],
    'A': [('D', 4), ('E', 2)],
    'B': [('F', 6)],
    'C': [('B', 2), ('G', 9)],
    'D': [('H', 3)],
    'E': [('G', 5)],
    'F': [('I', 4)],
    'H': [('G', 1)],
    'I': [('G', 2)],
    'G': []
}

START = 'S'
GOAL = 'G'

# Breadth-First Search (BFS)
def bfs(start, goal):
    frontier = deque()
    frontier.append((start, [start]))
    visited = set()
    expanded_count = 0

    while frontier:
        node, path = frontier.popleft()

        if node == goal:
            return path, expanded_count

        if node not in visited:
            visited.add(node)
            expanded_count += 1

            for child, cost in graph[node]:
                if child not in visited:
                    frontier.append((child, path + [child]))

    return None, expanded_count


# Depth-First Search (DFS)
def dfs(start, goal):
    frontier = []
    frontier.append((start, [start]))
    visited = set()
    expanded_count = 0

    while frontier:
        node, path = frontier.pop()

        if node == goal:
            return path, expanded_count

        if node not in visited:
            visited.add(node)
            expanded_count += 1

            # Reverse to maintain left-to-right order
            for child, cost in reversed(graph[node]):
                if child not in visited:
                    frontier.append((child, path + [child]))

    return None, expanded_count


# Uniform Cost Search (UCS)
def ucs(start, goal):
    frontier = []
    heapq.heappush(frontier, (0, start, [start]))
    visited = set()
    expanded_count = 0

    while frontier:
        cost, node, path = heapq.heappop(frontier)

        if node == goal:
            return path, cost, expanded_count

        if node not in visited:
            visited.add(node)
            expanded_count += 1

            for child, step_cost in graph[node]:
                if child not in visited:
                    heapq.heappush(frontier,
                                   (cost + step_cost,
                                    child,
                                    path + [child]))

    return None, None, expanded_count


# Depth-Limited Search (DLS)
def dls(node, goal, limit, path, visited, expanded_count):
    if node == goal:
        return path, expanded_count

    if limit == 0:
        return None, expanded_count

    visited.add(node)
    expanded_count += 1

    for child, cost in graph[node]:
        if child not in visited:
            result, expanded_count = dls(
                child,
                goal,
                limit - 1,
                path + [child],
                visited.copy(),
                expanded_count
            )
            if result is not None:
                return result, expanded_count

    return None, expanded_count


# Iterative Deepening Search (IDS)
def ids(start, goal, max_depth):
    total_expanded = 0

    for depth in range(max_depth + 1):
        visited = set()
        result, expanded = dls(start, goal, depth, [start], visited, 0)
        total_expanded += expanded

        if result is not None:
            return result, depth, total_expanded

    return None, None, total_expanded


if __name__ == "__main__":

    print("\n===== BFS =====")
    path, expanded = bfs(START, GOAL)
    print("Path:", path)
    print("Nodes Expanded:", expanded)

    print("\n===== DFS =====")
    path, expanded = dfs(START, GOAL)
    print("Path:", path)
    print("Nodes Expanded:", expanded)

    print("\n===== UCS =====")
    path, cost, expanded = ucs(START, GOAL)
    print("Path:", path)
    print("Total Cost:", cost)
    print("Nodes Expanded:", expanded)

    print("\n===== IDS =====")
    path, depth, expanded = ids(START, GOAL, max_depth=5)
    print("Path:", path)
    print("Depth Found:", depth)
    print("Total Nodes Expanded:", expanded)


===== BFS =====
Path: ['S', 'C', 'G']
Nodes Expanded: 7

===== DFS =====
Path: ['S', 'A', 'D', 'H', 'G']
Nodes Expanded: 4

===== UCS =====
Path: ['S', 'A', 'E', 'G']
Total Cost: 9
Nodes Expanded: 7

===== IDS =====
Path: ['S', 'C', 'G']
Depth Found: 2
Total Nodes Expanded: 5


# Comparative Analysis of Uninformed Search Algorithms

We compare BFS, DFS, UCS, DLS, and IDS based on completeness, optimality, time, and space.


## 1. Completeness, Optimality, Memory Efficiency

### Complete
- BFS  
- UCS (if step costs ≥ 0)  
- IDS  
- DFS (may go infinitely deep)  
- DLS (may miss solution beyond limit)

### Optimal
- UCS (guarantees lowest-cost solution)  
- BFS and IDS (only if all step costs are equal)  
- DFS and DLS (not optimal)

### Most Memory Efficient
- DFS  
- DLS  
- IDS  
- BFS and UCS use the most memory


## 2. Complexity Ranking

Let:
- b = branching factor  
- d = solution depth  
- m = maximum depth  
- L = depth limit  

### Time Complexity (Worst Case)
- BFS: O(b^d)  
- DFS: O(b^m)  
- UCS: O(b^(C*/ε))  
- DLS: O(b^L)  
- IDS: O(b^d)  

**Typical Ranking (Fast → Slow):**  
DFS → DLS → IDS → BFS → UCS  


### Space Complexity
- BFS: O(b^d)  
- DFS: O(bm)  
- UCS: O(b^d)  
- DLS: O(bL)  
- IDS: O(bd)  

**Memory Ranking (Low → High):**  
DFS → DLS → IDS → BFS → UCS  


## 3. Why UCS Over BFS for Non-Uniform Costs?

BFS expands shallowest nodes and ignores cost.  
If step costs differ, it may return a more expensive path.

UCS expands the node with the lowest cumulative cost first,  
therefore it always finds the optimal (least-cost) solution.


## 4. When Prefer DFS or DLS?

- When memory is limited  
- When solution is very deep  
- When depth limit is known  
- In backtracking problems (e.g., constraint satisfaction)


### Final Summary

- **Complete:** BFS, UCS, IDS  
- **Optimal:** UCS  
- **Most Memory Efficient:** DFS/DLS  
- **Best for Cost-Based Problems:** UCS  
- **Best for Low Memory Systems:** DFS/DLS  